# Tutorial -  VaR & CVaR
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

## 2. Obtención de datos históricos

In [ ]:
tickers = ["NU", "BAC", "JPM", "CIB", "C", "GS"]

# Precios y retornos
px = yf.download(tickers, start="2022-01-01", progress=False)["Close"]
# Calculate log returns
ret = np.log(px / px.shift(1)).dropna(how="all")

# Asegurar columnas en el orden de 'tickers'
ret = ret[tickers]

## VaR Hist

In [ ]:
def var_hist(series, alpha=0.95):
    # VaR como pérdida positiva: -cuantil inferior
    q = series.quantile(1 - alpha)
    return q

## CVaR Hist

In [ ]:
def cvar_hist(series, alpha=0.95):
    # Promedio de pérdidas más severas que el cuantil (Expected Shortfall)
    q = series.quantile(1 - alpha)
    tail = series[series <= q]
    return tail.mean()

## Tabla VaR & CVaR

In [ ]:
plt.rcParams["figure.dpi"] = 120

levels = [0.95, 0.99]

rows = []
for t in tickers:
    s = ret[t].dropna()
    vals = {
        "Ticker": t,
        "VaR_95": var_hist(s, 0.95),
        "CVaR_95": cvar_hist(s, 0.95),
        "VaR_99": var_hist(s, 0.99),
        "CVaR_99": cvar_hist(s, 0.99),
    }
    rows.append(vals)

res = pd.DataFrame(rows).set_index("Ticker")

# Tabla ordenada por riesgo extremo (CVaR_99)
res_sorted = res.sort_values("CVaR_99", ascending=False)

# Mostrar tabla (%)
display(
    res_sorted.map(lambda x: f"{100*x:.2f}%")
)


In [ ]:
# --- Gráficas (barras horizontales) ---
def economist_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)

def plot_bars(series, title):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    series.sort_values(ascending=False).plot(kind="barh", alpha=0.8, ax=ax)
    economist_axes(ax)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Pérdida diaria (%)")
    # Etiquetas al final de cada barra
    for i, v in enumerate(series.sort_values(ascending=False).values):
        ax.text(v, i, f" {100*v:.2f}%", va="center")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_bars(res["VaR_95"],  "VaR histórico 95% (desde 2020-01-01)")

In [ ]:
plot_bars(res["CVaR_95"], "CVaR histórico 95% (Expected Shortfall)")

In [ ]:
plot_bars(res["VaR_99"],  "VaR histórico 99% (desde 2020-01-01)")

In [ ]:
plot_bars(res["CVaR_99"], "CVaR histórico 99% (Expected Shortfall)")

## VaR histórico rolling

In [ ]:
win = 252

# --- VaR histórico rolling: pérdidas como valores positivos
# Cuantiles inferiores de retornos (colas negativas)
q05 = ret.rolling(win).quantile(0.05)
q01 = ret.rolling(win).quantile(0.01)

VaR95 = q05
VaR99 = q01

# Opcional: DataFrame combinado con columnas jerárquicas
VaR_series = pd.concat(
    {"VaR_95": VaR95, "VaR_99": VaR99},
    axis=1
).dropna(how="all")

# Últimos valores de cada ticker
print("Último VaR rolling (ventana 252 días):")
print(pd.concat({
    "VaR_95": VaR95.tail(1).T.squeeze(),
    "VaR_99": VaR99.tail(1).T.squeeze()
}, axis=1).applymap(lambda x: f"{100*x:.2f}%"))


In [ ]:
# 2) Graficar la serie de VaR 99% de un ticker (opcional)
ticker = "NU"
plt.figure(figsize=(9,4.5))
VaR99[ticker].dropna().plot()
plt.title(f"Serie VaR 99% (rolling 252 días) - {ticker}", fontsize=12, fontweight="bold")
plt.ylabel("Pérdida diaria (%)")
plt.xlabel("")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.box(False)
plt.ylim(-0.2, 0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Gráfica comparativa: todas las series VaR 95% ---
plt.figure(figsize=(10,6))

for t in tickers:
    plt.plot(VaR95[t], label=t, alpha=0.8)

plt.title("VaR histórico 95% (ventana móvil de 252 días)", fontsize=13, fontweight="bold")
plt.ylabel("Pérdida diaria (%)")
plt.xlabel("")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.legend()
plt.box(False)
plt.ylim(-0.1, 0)
plt.tight_layout()
plt.show()

El VaR histórico al 95% muestra que NU presenta el mayor riesgo de pérdida, con valores que llegan aproximadamente a −9,5% al inicio y se mantienen alrededor de −3,5% recientemente. Las demás acciones presentan un riesgo considerablemente menor y más estable, destacándose CIB y BAC por sus niveles relativamente moderados, mientras que JPM, C y GS se concentran generalmente entre −2% y −3%. En general, el riesgo de todas las acciones tiende a converger hacia niveles cercanos al −2% a −3% al final del período, aunque NU continúa siendo claramente la acción más riesgosa del portafolio.